# Student Performance Analysis
This notebook loads, processes, and analyzes student performance data from the provided CSV file.

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = '/content/data (1).csv'
df = pd.read_csv(file_path)

# Display basic information and first few rows
print("Dataset Schema and Info:")
display(df.info())
print("\nFirst 5 Rows:")
display(df.head())

Dataset Schema and Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Student_ID  40 non-null     int64  
 1   Name        40 non-null     object 
 2   Department  39 non-null     object 
 3   Math        39 non-null     float64
 4   Science     39 non-null     float64
 5   English     39 non-null     float64
 6   Attendance  39 non-null     float64
dtypes: float64(4), int64(1), object(2)
memory usage: 2.3+ KB


None


First 5 Rows:


,Student_ID,Name,Department,Math,Science,English,Attendance
0,101,Aarav Sharma,Computer Science,85.0,90.0,78.0,92.0
1,102,Priya Patel,Information Tech,72.0,65.0,80.0,85.0
2,103,Rohan Verma,Electronics,90.0,88.0,94.0,95.0
3,104,Ananya Singh,Computer Science,60.0,55.0,58.0,78.0
4,105,Vikram Malhotra,Mechanical,45.0,50.0,48.0,70.0


### Processing and Calculations
We will define reusable functions to:
1. Calculate total and average marks.
2. Assign grades and pass/fail status.
3. Analyze overall and subject-wise metrics.

In [ ]:
def calculate_grades_and_status(row, subject_cols, passing_score=40):
    """
    Calculates totals, averages, assigns grade, and determines pass/fail status.
    """
    marks = [row[sub] for sub in subject_cols if pd.notnull(row[sub])]
    if not marks:
        return 0, 0, 'F', 'Fail'

    total_marks = np.sum(marks)
    avg_marks = np.mean(marks)

    # Determine overall status (Fail if any academic subject is below passing_score)
    any_failed = any(row[sub] < passing_score for sub in subject_cols if pd.notnull(row[sub]))
    status = 'Fail' if any_failed else 'Pass'

    # Grading Rule
    if avg_marks >= 90:
        grade = 'A+'
    elif avg_marks >= 80:
        grade = 'A'
    elif avg_marks >= 70:
        grade = 'B'
    elif avg_marks >= 60:
        grade = 'C'
    elif avg_marks >= 50:
        grade = 'D'
    else:
        grade = 'F'
        status = 'Fail'

    return total_marks, avg_marks, grade, status

# Exclude non-academic and attendance columns
exclude_cols = ['id', 'name', 'student_id', 'roll_no', 'class', 'gender', 'age', 'attendance']
subject_cols = [col for col in df.columns if col.lower() not in exclude_cols and np.issubdtype(df[col].dtype, np.number)]

print(f"Detected academic subject columns for analysis: {subject_cols}")

Detected academic subject columns for analysis: ['Math', 'Science', 'English', 'Total Marks', 'Average Marks']


In [ ]:
# Apply calculations row by row
totals = []
averages = []
grades = []
statuses = []

for idx, row in df.iterrows():
    tot, avg, grd, stat = calculate_grades_and_status(row, subject_cols)
    totals.append(tot)
    averages.append(avg)
    grades.append(grd)
    statuses.append(stat)

df['Total Marks'] = totals
df['Average Marks'] = averages
df['Grade'] = grades
df['Status'] = statuses

print("Processed Student Data:")
display(df.head())

Processed Student Data:


,Student_ID,Name,Department,Math,Science,English,Attendance,Total Marks,Average Marks,Grade,Status
0,101,Aarav Sharma,Computer Science,85.0,90.0,78.0,92.0,345.0,86.25,A,Pass
1,102,Priya Patel,Information Tech,72.0,65.0,80.0,85.0,302.0,75.50,B,Pass
2,103,Rohan Verma,Electronics,90.0,88.0,94.0,95.0,367.0,91.75,A+,Pass
3,104,Ananya Singh,Computer Science,60.0,55.0,58.0,78.0,251.0,62.75,C,Pass
4,105,Vikram Malhotra,Mechanical,45.0,50.0,48.0,70.0,213.0,53.25,D,Pass


### Performance Metrics & Subject Analysis
Now we extract overall class stats and analyze performance by subject.

In [ ]:
# Overall Metrics
highest_avg = df['Average Marks'].max()
lowest_avg = df['Average Marks'].min()
class_avg = df['Average Marks'].mean()

print("=== Overall Class Performance Summary ===")
print(f"Highest Average Score: {highest_avg:.2f}%")
print(f"Lowest Average Score:  {lowest_avg:.2f}%")
print(f"Class Average Score:   {class_avg:.2f}%")
print("=========================================\n")

# Subject-wise Analysis
subject_summary = {}
for sub in subject_cols:
    subject_summary[sub] = {
        'Highest': df[sub].max(),
        'Lowest': df[sub].min(),
        'Average': round(df[sub].mean(), 2)
    }

subject_df = pd.DataFrame(subject_summary).T
print("Subject-wise Statistics:")
display(subject_df)

# Top performing student(s)
top_student_idx = df['Average Marks'].idxmax()
top_student = df.loc[top_student_idx]
print(f"\nTop Performing Student: {top_student.get('Name', top_student.get('name', 'N/A'))} with an Average of {top_student['Average Marks']:.2f}% (Grade: {top_student['Grade']})")

=== Overall Class Performance Summary ===
Highest Average Score: 95.75%
Lowest Average Score:  0.00%
Class Average Score:   75.39%

Subject-wise Statistics:


,Highest,Lowest,Average
Math,95.00,40.0,74.77
Science,95.00,45.0,74.54
English,96.00,42.0,76.03
Total Marks,383.00,0.0,301.58
Average Marks,95.75,0.0,75.39



Top Performing Student: Sneha Iyer with an Average of 95.75% (Grade: A+)
